In [3]:
smpls = ["beadpack", "bentheimer", "berea", "doddington", "estaillades", "gildehauser", "ketton", "portland"]
sizes = [2e-6, 3.0035e-6, 2.7745e-6, 2.6929e-6, 3.31136e-6, 4.4e-6, 3.00006e-6, 4.525e-6]
sample = smpls[2]  # choose sample
size = sizes[2]    # choose size

In [4]:
import simpleFoam_tools as sft
import pandas as pd
import numpy as np

In [5]:
remove = 1
if remove:
    sft.remove_run_files(f"{sample}")

In [4]:
domain_name = f"{sample}_z"  # "cylinder" or "vti_bent_x"

scale = size  # voxel size [m]
dp = 1.0           # pressure difference [Pa]
refinement = 1    # local refinement level
factor_mesh_x = 1.0    # global mesh scale factor
factor_mesh_y = 1.0    # global mesh scale factor
factor_mesh_z = 1.0    # global mesh scale factor

boundary_type = "symmetryPlane"  # "symmetryPlane" or "Wall"

dt = 1e-6
end_time = 50e-6
write_interval = 10

In [5]:
# Get voxel shape and a pore voxel location from VTI
vti_path = f"{sample}/constant/geometry/{domain_name}.vti"
shape = sft.vti_shape(vti_path)
location_in_mesh = sft.find_pore_location(vti_path)

# Convert VTI to STL surface mesh
stl = f"{domain_name}.stl"
stl_path = f"{sample}/constant/triSurface/{stl}"
sft.vti_to_stl(vti_path, stl_path)

# Adjust mesh resolution
mesh_resolution = (int(shape[0]*factor_mesh_x), int(shape[1]*factor_mesh_y), int(shape[2]*factor_mesh_z))

# File paths
blockMeshDict_path    = f"{sample}/system/blockMeshDict"
controlDict_path      = f"{sample}/system/controlDict"
p_field_path          = f"{sample}/0/p"
U_field_path          = f"{sample}/0/U"
snappyHexMeshDict_path = f"{sample}/system/snappyHexMeshDict"
# Generate OpenFOAM files
sft.generate_blockMeshDict(shape, mesh_resolution, blockMeshDict_path, boundary=boundary_type)
sft.generate_snappyHexMeshDict(location_in_mesh, stl, snappyHexMeshDict_path, refinement=refinement)
sft.generate_controlDict(controlDict_path, end_time=end_time, write_interval=write_interval, dt=dt)
sft.generate_pressure_field(p_field_path, dp=dp, boundary=boundary_type)
sft.generate_velocity_field(U_field_path, boundary=boundary_type)

Running command: pvpython /home/h09435ap/porePermFoam/simpleFoam_tools/paraview_stl.py /home/h09435ap/porePermFoam/berea/constant/geometry/berea_z.vti /home/h09435ap/porePermFoam/berea/constant/triSurface/berea_z.stl
hwloc/linux: Ignoring PCI device with non-16bit domain.
Pass --enable-32bits-pci-domain to configure to support such devices
(warning: it would break the library ABI, don't enable unless really needed).
✅ Wrote ASCII STL: /home/h09435ap/porePermFoam/berea/constant/triSurface/berea_z.stl
Generated blockMeshDict at: berea/system/blockMeshDict
Generated snappyHexMeshDict at: berea/system/snappyHexMeshDict
Generated controlDict at: berea/system/controlDict
Generated p at: berea/0/p with boundary symmetryPlane
Generated U at: berea/0/U with boundary symmetryPlane


In [6]:
sft.run_simplefoam(f"{sample}", scale=scale)


>>> blockMesh

/*---------------------------------------------------------------------------*\
| =========                 |                                                 |
| \\      /  F ield         | OpenFOAM: The Open Source CFD Toolbox           |
|  \\    /   O peration     | Version:  2412                                  |
|   \\  /    A nd           | Website:  www.openfoam.com                      |
|    \\/     M anipulation  |                                                 |
\*---------------------------------------------------------------------------*/
Build  : _e5c6ccc3-20250814 OPENFOAM=2412 patch=250814 version=2412
Arch   : "LSB;label=32;scalar=64"
Exec   : blockMesh
Date   : Feb 04 2026
Time   : 12:47:24
Host   : e-10aux32876g5
PID    : 252798
I/O    : uncollated
Case   : /home/h09435ap/porePermFoam/berea
nProcs : 1
trapFpe: Floating point exception trapping enabled (FOAM_SIGFPE).
fileModificationChecking : Monitoring run-time modified files using timeStampMaster 

In [8]:
df_q_area = pd.read_csv(f"{sample}/q_in.csv")
df_q_area.head()

,time,flowRate_phi,area
0,0.00000,-1.991810e-12,1.473820e-07
1,0.00001,-2.349010e-12,1.473820e-07
2,0.00002,-2.664400e-12,1.473820e-07
3,0.00003,-2.766960e-12,1.473820e-07
4,0.00004,-2.819270e-12,1.473820e-07


In [9]:
phi = sft.vti_phi(vti_path)
print(f"Porosity: {phi:.02%}")

# Cross-sectional area
A = shape[1] * shape[2] * scale**2
miu = 1e-3  # fluid viscosity [Pa·s]
L = shape[0] * scale  # length [m]

# Flow rate from last timestep
Q = np.abs(df_q_area["flowRate_phi"].iloc[-1])

# Darcy's law
k = Q * miu * L / (A * dp)

print(f"Permeability: {k:.02e} m^2")
print(f"Permeability: {k * 1.01324997e15:.02f} md")

Porosity: 22.06%
Permeability: 4.06e-12 m^2
Permeability: 4118.40 md
